# Statistical Significance Analysis
**Input:** 6 pkl files from 30-run experiments
**Tests:** Friedman + Wilcoxon post-hoc (Bonferroni correction)
**Note:**
- S1A pkl structure: `clustering_results_30[ds][method]`  
- S1B pkl structure: `clustering_results_30[ds][method]`  
- S2 pkl structure: `final_results_30[ds][method]`  
- S2A/S2B each split into kmf + ccf → must merge methods per dataset


## Cell 1: Imports

In [3]:
import pickle
import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, wilcoxon
from itertools import combinations
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import warnings; warnings.filterwarnings('ignore')
print("Imports OK")

Imports OK


## Cell 2: Load pkl Files

In [5]:
with open('Strategy1_30runs_PartA.pkl', 'rb') as f: s1a = pickle.load(f)
with open('Strategy1_30runs_PartB.pkl', 'rb') as f: s1b = pickle.load(f)
with open('S2A_results_kmf_30runs.pkl', 'rb') as f: s2a_kmf = pickle.load(f)
with open('S2A_results_ccf_30runs.pkl', 'rb') as f: s2a_ccf = pickle.load(f)
with open('S2B_results_kmf_30runs.pkl', 'rb') as f: s2b_kmf = pickle.load(f)
with open('S2B_results_ccf_30runs.pkl', 'rb') as f: s2b_ccf = pickle.load(f)
print("All loaded ✓")

All loaded ✓


## Cell 3: Extract & Merge Data

In [7]:
METHOD_LABEL = {
    'kmeans':              'K-Means',
    'fairlet':             'Fairlet',
    'bfkm':               'BFKM',
    'fair_centroid':       'CCF',
    'postprocessing_nfp':  'PP-NFP',
    'postprocessing_gini': 'PP-Gini',
    'rawlsian':            'Rawlsian',
}
METHODS   = list(METHOD_LABEL.keys())
M_LABELS  = list(METHOD_LABEL.values())

UTIL_COLS = ['Inertia','Silhouette','Calinski_Harabasz','Davies_Bouldin','BCSS','Variance_Ratio']
FAIR_COLS = ['Balance','SPD','Disparate_Impact','Entropy']
ALL_COLS  = UTIL_COLS + FAIR_COLS
HIGHER    = {'Silhouette','Calinski_Harabasz','BCSS','Variance_Ratio','Balance','Entropy'}

N_PAIRS = len(list(combinations(METHODS, 2)))   # 21
ALPHA   = 0.05
ALPHA_B = ALPHA / N_PAIRS                        # ≈ 0.00238

# ── Extract from S1 pkl ──────────────────────────────────────────────────────
def extract_s1(pkl):
    """
    S1A/S1B structure:
      pkl['clustering_results_30'][ds][method]
        → r['mean_quality'], r['mean_fairness']
    Old S1A may have ['clustering_results_30']['approach2'][ds][method]
    """
    results = pkl['clustering_results_30']
    # Detect old structure
    first_key = list(results.keys())[0]
    if first_key in ['approach1', 'approach2']:
        results = results.get('approach2', results.get('approach1', {}))
    out = {}
    for ds, methods in results.items():
        out[ds] = {}
        for m, r in methods.items():
            out[ds][m] = {**r['mean_quality'], **r['mean_fairness']}
    return out

# ── Extract from S2 pkl ──────────────────────────────────────────────────────
def extract_s2(pkl):
    """
    S2 structure: pkl['final_results_30'][ds][method]
      → r['mean_quality'], r['mean_fairness']
    """
    results = pkl['final_results_30']
    out = {}
    for ds, methods in results.items():
        out[ds] = {}
        for m, r in methods.items():
            out[ds][m.lower()] = {**r['mean_quality'], **r['mean_fairness']}
    return out

# ── Merge S1 ─────────────────────────────────────────────────────────────────
s1_data = {}
for pkl in [s1a, s1b]:
    for ds, methods in extract_s1(pkl).items():
        if ds not in s1_data:
            s1_data[ds] = {}
        s1_data[ds].update(methods)

# ── Merge S2 — must merge methods per dataset, not overwrite ─────────────────
s2_data = {}
for pkl in [s2a_kmf, s2a_ccf, s2b_kmf, s2b_ccf]:
    for ds, methods in extract_s2(pkl).items():
        if ds not in s2_data:
            s2_data[ds] = {}
        s2_data[ds].update(methods)   # merge methods, not overwrite dataset

print(f"S1 datasets: {len(s1_data)}  methods per ds: {len(list(s1_data.values())[0])}")
print(f"S2 datasets: {len(s2_data)}  methods per ds: {len(list(s2_data.values())[0])}")
print(f"N_pairs: {N_PAIRS}, Bonferroni α: {ALPHA_B:.5f}")

# Sanity check
first_s2 = list(s2_data.keys())[0]
print(f"\nS2 first dataset '{first_s2}' methods: {list(s2_data[first_s2].keys())}")

S1 datasets: 18  methods per ds: 7
S2 datasets: 18  methods per ds: 7
N_pairs: 21, Bonferroni α: 0.00238

S2 first dataset 'adult' methods: ['kmeans', 'fairlet', 'bfkm', 'fair_centroid', 'postprocessing_nfp', 'postprocessing_gini', 'rawlsian']


## Cell 4: Build Score Matrix

In [9]:
def build_score_matrix(data, metric):
    """
    DataFrame: rows=datasets, cols=methods.
    Only include datasets where ALL 7 methods have a value for this metric.
    """
    complete_ds = []
    for ds in data:
        has_all = all(
            m in data[ds] and metric in data[ds][m]
            and not np.isnan(data[ds][m][metric])
            for m in METHODS
        )
        if has_all:
            complete_ds.append(ds)

    if not complete_ds:
        return pd.DataFrame()

    rows = []
    for ds in complete_ds:
        row = {'Dataset': ds}
        for m in METHODS:
            row[m] = data[ds][m][metric]
        rows.append(row)
    return pd.DataFrame(rows).set_index('Dataset')

# Check coverage
print("=== Dataset coverage per metric (S1) ===")
for col in ALL_COLS:
    mat = build_score_matrix(s1_data, col)
    print(f"  {col:25s}: {len(mat)} complete datasets")

print("\n=== Dataset coverage per metric (S2) ===")
for col in ALL_COLS:
    mat = build_score_matrix(s2_data, col)
    print(f"  {col:25s}: {len(mat)} complete datasets")

=== Dataset coverage per metric (S1) ===
  Inertia                  : 18 complete datasets
  Silhouette               : 18 complete datasets
  Calinski_Harabasz        : 18 complete datasets
  Davies_Bouldin           : 18 complete datasets
  BCSS                     : 18 complete datasets
  Variance_Ratio           : 18 complete datasets
  Balance                  : 18 complete datasets
  SPD                      : 18 complete datasets
  Disparate_Impact         : 18 complete datasets
  Entropy                  : 18 complete datasets

=== Dataset coverage per metric (S2) ===
  Inertia                  : 18 complete datasets
  Silhouette               : 18 complete datasets
  Calinski_Harabasz        : 18 complete datasets
  Davies_Bouldin           : 18 complete datasets
  BCSS                     : 18 complete datasets
  Variance_Ratio           : 18 complete datasets
  Balance                  : 18 complete datasets
  SPD                      : 18 complete datasets
  Disparate_Impac

## Cell 5: Friedman Test

In [11]:
def run_friedman(data, metrics):
    rows = []
    for metric in metrics:
        mat = build_score_matrix(data, metric)
        if mat.empty or len(mat) < 3:
            rows.append({'Metric': metric, 'N_datasets': len(mat) if not mat.empty else 0,
                         'Friedman_stat': 'N/A', 'p_value': 'N/A', 'Significant': 'N/A'})
            continue
        arrays = [mat[m].values for m in METHODS]
        try:
            stat, p = friedmanchisquare(*arrays)
            sig = 'YES *' if p < ALPHA else 'no'
        except Exception:
            stat, p, sig = np.nan, np.nan, 'error'
        rows.append({
            'Metric':        metric,
            'N_datasets':    len(mat),
            'Friedman_stat': round(stat, 3) if not np.isnan(stat) else 'N/A',
            'p_value':       round(p, 6)    if not np.isnan(p)    else 'N/A',
            'Significant':   sig
        })
    return pd.DataFrame(rows)

friedman_s1 = run_friedman(s1_data, ALL_COLS)
friedman_s2 = run_friedman(s2_data, ALL_COLS)

print("=" * 65)
print("FRIEDMAN TEST — Strategy 1")
print("=" * 65)
print(friedman_s1.to_string(index=False))

print("\n" + "=" * 65)
print("FRIEDMAN TEST — Strategy 2")
print("=" * 65)
print(friedman_s2.to_string(index=False))

FRIEDMAN TEST — Strategy 1
           Metric  N_datasets  Friedman_stat  p_value Significant
          Inertia          18         82.024 0.000000       YES *
       Silhouette          18         59.470 0.000000       YES *
Calinski_Harabasz          18         82.024 0.000000       YES *
   Davies_Bouldin          18         62.072 0.000000       YES *
             BCSS          18         82.024 0.000000       YES *
   Variance_Ratio          18         82.024 0.000000       YES *
          Balance          18         45.180 0.000000       YES *
              SPD          18         35.859 0.000003       YES *
 Disparate_Impact          18         34.098 0.000006       YES *
          Entropy          18         15.309 0.017986       YES *

FRIEDMAN TEST — Strategy 2
           Metric  N_datasets  Friedman_stat  p_value Significant
          Inertia          18         80.988 0.000000       YES *
       Silhouette          18         50.241 0.000000       YES *
Calinski_Harabasz    

## Cell 6: Wilcoxon Post-hoc + Bonferroni

In [13]:
def run_wilcoxon(data, metric):
    mat = build_score_matrix(data, metric)
    if mat.empty:
        return pd.DataFrame()
    pairs = list(combinations(METHODS, 2))
    rows  = []
    for m1, m2 in pairs:
        x = mat[m1].values
        y = mat[m2].values
        n = min(len(x), len(y))
        if n < 3:
            rows.append({'Method_A': METHOD_LABEL[m1], 'Method_B': METHOD_LABEL[m2],
                         'W_stat': '-', 'p_raw': '-', 'p_bonf': '-',
                         f'Sig(p<{ALPHA_B:.4f})': '-'})
            continue
        try:
            stat, p   = wilcoxon(x[:n], y[:n], zero_method='wilcox')
            p_bonf    = min(p * N_PAIRS, 1.0)
            sig       = 'YES *' if p < ALPHA_B else 'no'
        except Exception:
            stat = p = p_bonf = np.nan; sig = '-'
        rows.append({
            'Method_A':              METHOD_LABEL[m1],
            'Method_B':              METHOD_LABEL[m2],
            'W_stat':                round(stat, 3)   if not np.isnan(stat)   else '-',
            'p_raw':                 round(p, 6)      if not np.isnan(p)      else '-',
            'p_bonf':                round(p_bonf, 6) if not np.isnan(p_bonf) else '-',
            f'Sig(p<{ALPHA_B:.4f})': sig
        })
    return pd.DataFrame(rows)

# Run for significant metrics only
wilcoxon_s1, wilcoxon_s2 = {}, {}

for _, row in friedman_s1.iterrows():
    if str(row['Significant']).startswith('YES'):
        wilcoxon_s1[row['Metric']] = run_wilcoxon(s1_data, row['Metric'])

for _, row in friedman_s2.iterrows():
    if str(row['Significant']).startswith('YES'):
        wilcoxon_s2[row['Metric']] = run_wilcoxon(s2_data, row['Metric'])

print(f"S1 significant metrics ({len(wilcoxon_s1)}): {list(wilcoxon_s1.keys())}")
print(f"S2 significant metrics ({len(wilcoxon_s2)}): {list(wilcoxon_s2.keys())}")

if wilcoxon_s1:
    m = list(wilcoxon_s1.keys())[0]
    print(f"\nExample — S1 Wilcoxon post-hoc for {m}:")
    print(wilcoxon_s1[m].to_string(index=False))

S1 significant metrics (10): ['Inertia', 'Silhouette', 'Calinski_Harabasz', 'Davies_Bouldin', 'BCSS', 'Variance_Ratio', 'Balance', 'SPD', 'Disparate_Impact', 'Entropy']
S2 significant metrics (10): ['Inertia', 'Silhouette', 'Calinski_Harabasz', 'Davies_Bouldin', 'BCSS', 'Variance_Ratio', 'Balance', 'SPD', 'Disparate_Impact', 'Entropy']

Example — S1 Wilcoxon post-hoc for Inertia:
Method_A Method_B  W_stat    p_raw   p_bonf Sig(p<0.0024)
 K-Means  Fairlet     0.0 0.000008 0.000160         YES *
 K-Means     BFKM     1.0 0.000015 0.000320         YES *
 K-Means      CCF     0.0 0.000008 0.000160         YES *
 K-Means   PP-NFP     0.0 0.000655 0.013754         YES *
 K-Means  PP-Gini     0.0 0.000655 0.013754         YES *
 K-Means Rawlsian     1.0 0.000015 0.000320         YES *
 Fairlet     BFKM     0.0 0.000008 0.000160         YES *
 Fairlet      CCF     0.0 0.000008 0.000160         YES *
 Fairlet   PP-NFP     0.0 0.000008 0.000160         YES *
 Fairlet  PP-Gini     0.0 0.000008 0.

## Cell 7: Win-Loss Count

In [15]:
def win_counts(data, metrics_list, higher_set):
    """Count wins per method across all datasets × metrics."""
    wins  = {METHOD_LABEL[m]: 0 for m in METHODS}
    total = 0
    for metric in metrics_list:
        mat = build_score_matrix(data, metric)
        if mat.empty:
            continue
        for ds in mat.index:
            row_vals = {m: mat.loc[ds, m] for m in METHODS
                        if not np.isnan(mat.loc[ds, m])}
            if not row_vals:
                continue
            winner = max(row_vals, key=row_vals.get) if metric in higher_set                      else min(row_vals, key=row_vals.get)
            wins[METHOD_LABEL[winner]] += 1
            total += 1
    return wins, total

wins_s1, tot_s1 = win_counts(s1_data, ALL_COLS, HIGHER)
wins_s2, tot_s2 = win_counts(s2_data, ALL_COLS, HIGHER)

win_df = pd.DataFrame({
    'Method':  list(wins_s1.keys()),
    'S1_Wins': list(wins_s1.values()),
    'S1_%':    [f"{v/tot_s1*100:.1f}%" if tot_s1 > 0 else '-' for v in wins_s1.values()],
    'S2_Wins': [wins_s2[m] for m in wins_s1],
    'S2_%':    [f"{wins_s2[m]/tot_s2*100:.1f}%" if tot_s2 > 0 else '-' for m in wins_s1],
})
print(f"Total comparisons: S1={tot_s1}, S2={tot_s2}")
print(win_df.to_string(index=False))

Total comparisons: S1=180, S2=180
  Method  S1_Wins  S1_%  S2_Wins  S2_%
 K-Means       88 48.9%       82 45.6%
 Fairlet       47 26.1%       47 26.1%
    BFKM       18 10.0%       18 10.0%
     CCF        5  2.8%       13  7.2%
  PP-NFP       10  5.6%        9  5.0%
 PP-Gini        0  0.0%        0  0.0%
Rawlsian       12  6.7%       11  6.1%


## Cell 8: Save to Excel

In [17]:
wb   = Workbook()
wb.remove(wb.active)
thin = Side(style='thin', color='CCCCCC')
bdr  = Border(left=thin, right=thin, top=thin, bottom=thin)

def df_to_ws(wb, df, sheet_name, highlight_sig=True):
    ws = wb.create_sheet(sheet_name[:31])
    hf  = PatternFill('solid', start_color='2E74B5')
    sig_fill = PatternFill('solid', start_color='FFF2CC')
    for ci, col in enumerate(df.columns, 1):
        c = ws.cell(1, ci, col)
        c.font      = Font(bold=True, color='FFFFFF', size=9, name='Arial')
        c.fill      = hf
        c.alignment = Alignment(horizontal='center', wrap_text=True)
        c.border    = bdr
        ws.column_dimensions[get_column_letter(ci)].width = max(14, len(str(col))+2)
    for ri, row in enumerate(df.itertuples(index=False), 2):
        is_sig = any('YES' in str(v) for v in row)
        for ci, val in enumerate(row, 1):
            c = ws.cell(ri, ci, val)
            c.font      = Font(size=9, name='Arial',
                               bold=('YES' in str(val)))
            c.fill      = sig_fill if (highlight_sig and is_sig) else PatternFill()
            c.alignment = Alignment(horizontal='center')
            c.border    = bdr
    ws.freeze_panes = 'A2'
    return ws

# Friedman sheets
df_to_ws(wb, friedman_s1, 'Friedman_S1')
df_to_ws(wb, friedman_s2, 'Friedman_S2')

# Win-loss
df_to_ws(wb, win_df, 'Win_Loss', highlight_sig=False)

# Wilcoxon sheets
for metric, df in wilcoxon_s1.items():
    if not df.empty:
        df_to_ws(wb, df, f'W_S1_{metric[:15]}')
for metric, df in wilcoxon_s2.items():
    if not df.empty:
        df_to_ws(wb, df, f'W_S2_{metric[:15]}')

wb.save('Statistical_Tests.xlsx')
print("✓ Saved: Statistical_Tests.xlsx")
print(f"  Sheets: Friedman_S1, Friedman_S2, Win_Loss + {len(wilcoxon_s1)} S1 Wilcoxon + {len(wilcoxon_s2)} S2 Wilcoxon")

# Also save pkl
with open('statistical_results.pkl', 'wb') as f:
    pickle.dump({
        'friedman_s1':  friedman_s1,
        'friedman_s2':  friedman_s2,
        'wilcoxon_s1':  wilcoxon_s1,
        'wilcoxon_s2':  wilcoxon_s2,
        'win_loss':     win_df,
        'alpha_bonf':   ALPHA_B,
        'n_pairs':      N_PAIRS,
    }, f)
print("✓ Saved: statistical_results.pkl")

✓ Saved: Statistical_Tests.xlsx
  Sheets: Friedman_S1, Friedman_S2, Win_Loss + 10 S1 Wilcoxon + 10 S2 Wilcoxon
✓ Saved: statistical_results.pkl
